# Baseline driver

Shared run cells select the named checkpoint and bounded-short or whole-recording protocol. Model runs default off. Missing or excluded inputs remain pending.


In [ ]:
import subprocess
import sys

INSTALL_RUNTIME = False
if INSTALL_RUNTIME:
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'transformers>=4.46.3,<5', 'faster-whisper==1.2.1',
                    'ctranslate2>=4.4,<5', 'librosa', 'soundfile'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e',
                    '/kaggle/working/shono'], check=True)


In [ ]:
from pathlib import Path

from shono.data import LicenseRegistry
from shono.protocol import execute_matrix, run_matrix
from shono.provenance import seed_everything

REPO = Path('/kaggle/working/shono')
AUDIO_ROOT = Path('/kaggle/input')
RUN_MODELS = False
registry = LicenseRegistry.load(REPO / 'data/licenses.json')
seed_everything(0)
entries = tuple(entry for entry in run_matrix()
                if entry.slug in {'large-v3', 'large-v3-turbo', 'tugstugi-medium'})

from shono.transcribe import (
    FasterWhisperTranscriber, LongFormPipelineTranscriber, LongFormTranscriber,
    ShortFormWhisperTranscriber, SileroVAD,
)
from shono.transcribe.convert import to_ct2

CT2_ROOT = Path('/kaggle/working/shono-ct2')
LOCAL_CHECKPOINTS = {}  # Explicit slug -> downloaded local Whisper checkpoint.
CONVERTED_MODELS = {}


def local_transcriber(entry):
    if entry.inference_mode == 'bounded-short':
        return ShortFormWhisperTranscriber(entry.model_id, language='bn', device='cuda')
    checkpoint = LOCAL_CHECKPOINTS.get(entry.slug)
    if checkpoint is None and entry.slug == 'shono-medium':
        checkpoint = entry.model_id
    if checkpoint is None or not Path(checkpoint).is_dir():
        raise ValueError('Declare an existing local checkpoint for ' + entry.slug)
    converted = CT2_ROOT / entry.slug
    key = (entry.model_id, str(Path(checkpoint).resolve()))
    if key not in CONVERTED_MODELS:
        # Rebuild scratch conversion once per checkpoint in this notebook run.
        to_ct2(checkpoint, converted, quantization='float16', force=True)
        CONVERTED_MODELS[key] = str(converted)
    pipeline = LongFormTranscriber(
        SileroVAD(), FasterWhisperTranscriber(str(converted), language='bn', device='cuda'),
        max_chunk_s=30.0, pad_s=0.25,
    )
    return LongFormPipelineTranscriber(pipeline)


## Run declared baseline cells

Read/code-switch windows are bounded to 30 seconds. Long-form entries use whole recordings and the same VAD/chunk settings for each Whisper system. The project base differs from the regional checkpoint cited by the Loop paper.


In [ ]:
outcomes = execute_matrix(
    entries, base_dir=REPO, audio_root=AUDIO_ROOT, registry=registry,
    transcriber_for=local_transcriber, run_models=RUN_MODELS,
    inference_settings={
        entry.slug + ':' + entry.inference_mode:
        ({'device': 'cuda', 'language': 'bn', 'task': 'transcribe',
          'max_window_s': 30.0} if entry.inference_mode == 'bounded-short' else
         {'device': 'cuda', 'language': 'bn', 'max_chunk_s': 30.0, 'pad_s': 0.25,
          'quantization': 'float16', 'compute_type': 'float16',
          'local_checkpoint': str(LOCAL_CHECKPOINTS.get(entry.slug, 'unknown')),
          'condition_on_previous_text': False, 'word_timestamps': True,
          'vad_threshold': 0.5, 'vad_min_speech_ms': 250})
        for entry in entries},
)
for outcome in outcomes:
    print(outcome.entry.system, outcome.entry.slice_name, outcome.status, outcome.reason)


## Export reports

Predictions and reports go to `outputs/`. Reports retain per-cell policy, input hashes and inference context. Pending cells carry no score.


In [ ]:
for name in ['baselines', 'longform', 'codeswitch', 'full']:
    subprocess.run([sys.executable, '-m', 'shono.eval', '--report', name],
                   cwd=REPO, check=True)
